In [29]:
##### Assembles origional capital and labor intensities in correct scale for R2's later
# for countries and sub-national regions 

import os
import numpy as np
import pandas as pd
import geopandas as gpd

In [30]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

### Load sub-national data 
sub_capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
sub_labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

### Load country data
country = pd.read_csv(f"{cd}/Data/Clean/Intensities/country_intensities.csv")

# Save_path
save_path = f"{cd}/Data/Clean/Intensities/intensities_for_eval.csv"

In [31]:
##### Prep sub-national data

# merge capital and labor
sub_national = pd.merge(sub_capital, sub_labor, on='PROJ_ID', how='outer')

# fix scale and add log units
sub_national['region_capital_intensity_USD_per_million_tonne'] = sub_national['capital_intensity_USD_per_tonne'] * 1e6
sub_national['log_region_capital_intensity_USD_per_million_tonne'] = np.log1p(sub_national['region_capital_intensity_USD_per_million_tonne'])
sub_national['region_labor_intensity_jobs_per_million_tonne'] = sub_national['labor_intensity_jobs_per_tonne'] * 1e6
sub_national['log_region_labor_intensity_jobs_per_million_tonne'] = np.log1p(sub_national['region_labor_intensity_jobs_per_million_tonne'])

# filter columns
col_to_keep = ['PROJ_ID', 
               'region_capital_intensity_USD_per_million_tonne',
               'log_region_capital_intensity_USD_per_million_tonne', 
               'region_labor_intensity_jobs_per_million_tonne', 
               'log_region_labor_intensity_jobs_per_million_tonne']

sub_national = sub_national[col_to_keep]

sub_national['country_ID'] = sub_national['PROJ_ID'].str[:3]

In [32]:
##### Prep country data
country['country_ID'] = country['ISO3']

# fix scale and add log units
country['country_capital_intensity_USD_per_million_tonne'] = country['capital_intensity_USD_per_tonne'] * 1e6
country['log_country_capital_intensity_USD_per_million_tonne'] = np.log1p(country['country_capital_intensity_USD_per_million_tonne'])
country['country_labor_intensity_jobs_per_million_tonne'] = country['labor_intensity_jobs_per_tonne'] * 1e6
country['log_country_labor_intensity_jobs_per_million_tonne'] = np.log1p(country['country_labor_intensity_jobs_per_million_tonne'])

col_to_keep = ['country_ID', 
               'country_capital_intensity_USD_per_million_tonne',
               'log_country_capital_intensity_USD_per_million_tonne', 
               'country_labor_intensity_jobs_per_million_tonne', 
               'log_country_labor_intensity_jobs_per_million_tonne']

country = country[col_to_keep]


In [33]:
##### Merge sub-national and country 

final = sub_national.merge(country, on='country_ID', how='left')

final.to_csv(save_path, index=False)